In [1]:
import sys
from pathlib import Path

# Notebook ka folder
NOTEBOOK_DIR = Path().resolve()

# Project root = parent folder
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Add project root to import path
sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

import warnings as w
w.filterwarnings("ignore")

Project root: D:\Langchain_LangGraph_03-12-2025\MyProject


## Below code we used to chunk document and stored into vector database

In [2]:
"""
Production-Grade RAG System with:
1. Hybrid Search (Semantic + Keyword)
2. Reranking (Cohere)
3. LangGraph Integration
4. Evaluation Metrics
5. Complete Pipeline

Run in Jupyter Notebook
"""


'\nProduction-Grade RAG System with:\n1. Hybrid Search (Semantic + Keyword)\n2. Reranking (Cohere)\n3. LangGraph Integration\n4. Evaluation Metrics\n5. Complete Pipeline\n\nRun in Jupyter Notebook\n'

In [ ]:
'''
PDF Folder
   |
   v
PDF Parsing Layer
   |
   |---> Text Extraction (pdfplumber)
   |         |
   |         v
   |     Text Chunking
   |         |
   |         v
   |     LLM Text Summary
   |
   |---> Image Extraction (PyMuPDF)
   |         |
   |         v
   |     BLIP Caption (raw)
   |         |
   |         v
   |     🔥 enrich_image_caption_dynamic
   |
   |---> Table Extraction (Camelot)
             |
             v
         Table → Markdown
             |
             v
         LLM Table Summary

(All modalities converted to TEXT)
   |
   v
OpenAI Embeddings (1536 dim)
   |
   v
BM25 Encoder (Sparse)
   |
   v
Pinecone Vector Store
(Dense + Sparse + Metadata)


#Retrieval Document.

User Question
   |
   v
Intent Detection
(text / image / table)
   |
   v
Hybrid Retrieval
(Dense Embeddings + BM25)
   |
   v
Top-20 Candidates
   |
   v
Cohere Reranker
(Top-5 best context)
   |
   v
Context Formatting
[TYPE: IMAGE | PAGE: x]
[TYPE: TABLE | PAGE: y]
[TYPE: TEXT | PAGE: z]
   |
   v
LLM (GPT-3.5 / GPT-4)
   |
   v
Final Answer (Text Explanation)

'''

#### LangChain related Library

In [3]:
#now importing all the Module which is used to build the AI Model.
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
from exception import CustomException
from logger_config import logger
import os,sys

#using openai chat model and embedding models
from langchain_openai import ChatOpenAI,OpenAIEmbeddings

#using groq chat model 
from langchain_groq import ChatGroq

#using open source chat model from hugging Face
from langchain_huggingface import ChatHuggingFace,HuggingFaceEmbeddings,HuggingFaceEndpoint

from config import *

from langchain_core.runnables import RunnableBranch,RunnableLambda,RunnableParallel,RunnableSequence,RunnablePassthrough

[2025-12-29 15:58:19,105]-config_variable.py-INFO -Loading the environment Variable
[2025-12-29 15:58:19,113]-config_variable.py-INFO -Environment Variable successfully Loaded


In [4]:
%pwd

'd:\\Langchain_LangGraph_03-12-2025\\MyProject\\notebooks'

#### LanGraph related Library

In [5]:
#import Langgraph related Modules
import langgraph
from langgraph.graph import StateGraph,START,END
from dataclasses import dataclass
from typing import TypedDict
from typing import Literal,List,Annotated,Dict
from langchain_core.messages import AnyMessage,AIMessage,HumanMessage,ToolMessage

from pydantic import BaseModel #using this class we can perform validation to schema

from langgraph.prebuilt import tool_node,tools_condition #in this class we put all tools together
#tools_condition wrt to tool msg it will route the flow data to ttol node to perform execution

from langchain_core.tools import tool,Tool,StructuredTool

from langgraph.graph.message import BaseMessage #this is special class which hold every mesaage init.



## step:1) defining the models components

In [6]:
model1 = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.2 #we call as creative parameter
)
model1

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000026EDDCDCBB0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000026EDDE6DD60>, root_client=<openai.OpenAI object at 0x0000026EDDCDC310>, root_async_client=<openai.AsyncOpenAI object at 0x0000026EDDE6DDC0>, temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [7]:
model2 = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2 #we call as creative parameter
)
model2

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000026EDE5FFE80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026EDE612940>, model_name='llama-3.1-8b-instant', temperature=0.2, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
llm = HuggingFaceEndpoint(  
repo_id="meta-llama/Llama-3.1-8B-Instruct",  
task="text-generation",  
max_new_tokens=512,  
do_sample=False,  
repetition_penalty=1.03,  
)  

model3 = ChatHuggingFace(llm=llm, verbose=True)
model3

ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id='meta-llama/Llama-3.1-8B-Instruct', repetition_penalty=1.03, stop_sequences=[], server_kwargs={}, model_kwargs={}, model='meta-llama/Llama-3.1-8B-Instruct', client=<InferenceClient(model='meta-llama/Llama-3.1-8B-Instruct', timeout=120)>, async_client=<InferenceClient(model='meta-llama/Llama-3.1-8B-Instruct', timeout=120)>, task='text-generation'), model_id='meta-llama/Llama-3.1-8B-Instruct', model_kwargs={})

In [9]:
### Hugging face Embedding Models.
from langchain_huggingface import HuggingFaceEmbeddings,HuggingFaceEndpointEmbeddings
hug_emb_model = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-large-en-v1.5",
    task = "feature-extraction",
)
hug_emb_model

HuggingFaceEndpointEmbeddings(client=<InferenceClient(model='BAAI/bge-large-en-v1.5', timeout=None)>, async_client=<InferenceClient(model='BAAI/bge-large-en-v1.5', timeout=None)>, model='BAAI/bge-large-en-v1.5', provider=None, repo_id='BAAI/bge-large-en-v1.5', task='feature-extraction', model_kwargs=None, huggingfacehub_api_token=None)

In [10]:
from langchain_openai import OpenAIEmbeddings
emb_model = OpenAIEmbeddings(
    model="text-embedding-3-small"  
)
emb_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x0000026EDE6D96A0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x0000026EBA65EA60>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [11]:
import torch
torch.cuda.is_available()

True

### calling hugging face llm model that will give description about image

In [12]:
import requests
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration


In [13]:
# processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
# model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large").to("cuda")

In [14]:
# conditional image captioning
# text = "a photography of"
# inputs = processor(raw_image, text, return_tensors="pt").to("cuda")

# out = model.generate(**inputs)
# print(processor.decode(out[0], skip_special_tokens=True))

In [15]:
from transformers import pipeline #most suitable way.
image_captioning_model = pipeline("image-to-text", model="Salesforce/blip-image-captioning-large")
image_captioning_model

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


#### Building MultiModel RAG workflow using LangChain

In [16]:
import pdfplumber
import fitz
from PIL import Image
import camelot
import pytesseract

# If Tesseract is not in PATH (Windows)
pytesseract.pytesseract.tesseract_cmd = r"C:\Users\Admin\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"

In [17]:
from typing import Any

In [ ]:
# BLOCK 1: PDF TEXT EXTRACTION (pdfplumber)
# ============================================
from typing import List, Dict, Any
import pdfplumber


def extract_text_from_pdf(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Extract page-wise text from PDF using pdfplumber
    (Better layout preservation than PyMuPDF)

    Args:
        pdf_path (str): PDF file path

    Returns:
        List[Dict]: Each dict contains page number, text content, and metadata
    """

    text_chunks = []
    page_text_map = {}

    # pdfplumber context manager (industry standard)
    with pdfplumber.open(pdf_path) as pdf:
        
        total_pages = len(pdf.pages)
        logger.info(f"📄 Total pages found: {total_pages}")

        for page_num, page in enumerate(pdf.pages, start=1):

            # Extract text with layout awareness
            text = page.extract_text(
                x_tolerance=2,   # horizontal spacing control
                y_tolerance=3,   # vertical spacing / line gap
                layout=True      # preserve page geometry
            )

            # Skip empty pages
            if text and text.strip():
                clean_text = text.strip()
                page_text_map[page_num] = clean_text
                
                text_chunks.append({
                    "type": "text",
                    "content": clean_text,
                    "page_number": page_num,
                    "source": pdf_path
                })

    print(f"✅ Extracted text from {len(text_chunks)} pages")
    return text_chunks, page_text_map


In [19]:
# ============================================
# BLOCK 2: IMAGE EXTRACTION FROM PDF
# ============================================
def extract_images_from_pdf(pdf_path: str, output_folder: str = r"D:\Langchain_LangGraph_03-12-2025\ImageDir") -> List[Dict[str, Any]]:
    """
    Extracting Image from Each Page Of PDF.
    
    Args:
        pdf_path: PDF file path
        output_folder: Images save karne ke liye folder
    
    Returns:
        List of dictionaries containing image info
    """
    # Output folder create karein agar exist nahi karta
    Path(output_folder).mkdir(exist_ok=True)
    
    image_chunks = []
    doc = fitz.open(pdf_path)
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        image_list = page.get_images()
        
        for img_index, img in enumerate(image_list):
            #xref is the unique id for image
            xref = img[0]
            
            #wrt to image ref unique ID extracting image from pdf
            base_image = doc.extract_image(xref)
            
            # Extract raw image bytes from the PDF image object
            image_bytes = base_image["image"]
            
            # Image save karein
            image_filename = f"{output_folder}/page{page_num + 1}_img{img_index + 1}.png"
            with open(image_filename, "wb") as img_file:
                img_file.write(image_bytes)
            
            image_chunks.append({
                'type': 'image',
                'image_path': image_filename,
                'page_number': page_num + 1,
                'source': pdf_path
            })
    
    doc.close()
    print(f"✅ Extracted {len(image_chunks)} images")
    return image_chunks

In [20]:
# ============================================
# BLOCK 3: TABLE EXTRACTION FROM PDF
# ============================================

from pathlib import Path
import camelot
import sys

def extract_tables_from_pdf(
    pdf_path: str,
    output_folder: str = r"D:\Langchain_LangGraph_03-12-2025\TableDir"
) -> List[Dict[str, Any]]:
    """
    Extract tables from PDF using Camelot (lattice + stream).
    Tables are converted into Markdown for RAG-friendly embeddings.

    Args:
        pdf_path (str): PDF file path
        output_folder (str): Directory to save extracted tables as markdown

    Returns:
        List[Dict]: Each dict contains table content (markdown) + metadata
    """

    table_chunks = []

    # Ensure output directory exists
    output_dir = Path(output_folder)
    output_dir.mkdir(parents=True, exist_ok=True)

    logger.info("📊 Extracting tables using Camelot (lattice + stream)")

    # 1️⃣ LATTICE → bordered / ruled tables
    lattice_tables = camelot.read_pdf(
        pdf_path,
        pages="all",
        flavor="lattice"
    )

    # 2️⃣ STREAM → unruled / research-style tables
    stream_tables = camelot.read_pdf(
        pdf_path,
        pages="all",
        flavor="stream"
    )

    # Combine both detections
    all_tables = list(lattice_tables) + list(stream_tables)
    logger.info(f"🔍 Total tables detected (raw): {len(all_tables)}")

    # Iterate over detected tables
    for idx, table in enumerate(all_tables, start=1):
        try:
            #converting into dataframe object
            df = table.df

            # Skip noise / invalid tables
            if df.empty or df.shape[0] < 2 or df.shape[1] < 2:
                continue

            # Convert DataFrame → Markdown (best for RAG)
            markdown_table = df.to_markdown(index=False)

            # Create markdown file name
            md_file = output_dir / f"table_page_{table.page}_idx_{idx}.md"

            # Write markdown to disk
            md_file.write_text(markdown_table, encoding="utf-8")

            # Store metadata + content
            table_chunks.append({
                "type": "table",
                "content": markdown_table,
                "page_number": table.page,
                "source": pdf_path,
                "rows": df.shape[0],
                "columns": df.shape[1],
                "flavor": table.flavor,
                "file_path": str(md_file)
            })

        except Exception as e:
            logger.info(f"❌ Error extracting table {idx}: {e}")
            raise CustomException(e, sys)

    logger.info(f"✅ Extracted {len(table_chunks)} valid tables")
    return table_chunks


In [21]:
# ============================================
# BLOCK 4: Generating RAW IMAGE CAPTION (OPEN-SOURCE)
# ============================================
def generate_image_description_opensource(image_path: str) -> str:
    """
    Hugging Face open-source model use karke image ka description generate karein
    
    Args:
        image_path: Image file ka path
    
    Returns:
        Image description text
    """
    try:
        # Image load karein
        image = Image.open(image_path)
        
        # BLIP model se caption generate karein
        result = image_captioning_model(image)
        description = result[0]['generated_text']
        
        logger.info(f"✅ Generated description for {image_path}")
        return description
    except Exception as e:
        raise CustomException(e,sys)
        return "Image description unavailable"
    
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()


In [22]:
# ============================================
# BLOCK : DYNAMIC IMAGE CAPTION ENRICHMENT (CORE FIX)
# ============================================
from langchain.prompts import ChatPromptTemplate

def enrich_image_caption_dynamic(
    raw_caption: str,
    page_number: int,
    page_text_map: dict
) -> str:

    nearby_texts = []
    for p in [page_number - 1, page_number, page_number + 1]:
        if p in page_text_map:
            nearby_texts.append(page_text_map[p][:1500])

    context = "\n\n".join(nearby_texts) or "No surrounding text available."

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You enrich image descriptions using ONLY provided text. "
         "Do NOT guess or add new concepts or hallucinate."
        ),
        ("human",
         """
Surrounding text:
{context}

Raw image caption:
{caption}

Task:
- Explain what the diagram represents
- Mention components ONLY if present in text
- If unclear, stay generic

Final description:
""")
    ])

    chain = prompt | model1 | parser

    return chain.invoke({
        "context": context,
        "caption": raw_caption
    })


In [23]:
# ============================================
# BLOCK 5: Generating summary for TEXT PROCESSING WITH CLOSED-SOURCE MODEL
# ============================================
def process_text_with_closedmodel(text: str) -> str:
    """
    OpenAI closed-source model use karke text ko process/summarize karein
    
    Args:
        text: Input text
    
    Returns:
        Processed/summarized text
    """
    try:
        # OpenAI API call karke text summarize karein
        text_prompt = ChatPromptTemplate.from_messages([
            ("system", "Summarize research paper text for semantic retrieval."),
            ("human", "{text}")
        ])

        text_chain = text_prompt | model1 | parser
        
        #invoking the chain to execute the task.
        processed_text = text_chain.invoke(input={'text':text})
        
        logger.info(f"✅ Processed text with closed-source model")
        return processed_text
    except Exception as e:
        raise CustomException(e,sys)
        return text  # Agar error aaye to original text return karein

In [24]:
# BLOCK 6: Generating Summary TABLE PROCESSING WITH CLOSED-SOURCE MODEL
# ============================================
def process_table_with_closedmodel(table_text: str) -> str:
    """
    OpenAI closed-source model use karke table ko process/describe karein
    
    Args:
        table_text: Table text
    
    Returns:
        Table description/summary
    """
    try:
        # OpenAI API call karke table describe karein
        table_prompt = ChatPromptTemplate.from_messages([
            ("system", "Explain the table, its structure, and key insights."),
            ("human", "{table_text}")
        ])

        table_chain = table_prompt | model1 | parser
        
        #invoking chain to execute the task.
        table_description = table_chain.invoke(input={'table_text':table_text})

        logger.info(f"✅ Processed table with closed-source model")
        return table_description
    except Exception as e:
        raise CustomException(e,sys)
        return table_text


In [25]:
# BLOCK 7: EMBEDDING GENERATION
# ============================================
def generate_embeddings(text: str) -> List[float]:
    """
    Sentence Transformer use karke text ke embeddings generate karein or 
    else i am using openai closed source model
    
    Args:
        text: Input text
    
    Returns:
        Embedding vector (list of floats)
    """
    # Sentence Transformer se embedding generate karein
    embedding = emb_model.embed_query(text=text)
    
    return embedding

In [26]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [27]:
# BLOCK 8: CHUNK CREATION AND PROCESSING(CREATE MULTIMODAL CHUNKS)
# ============================================
def create_document_chunks(pdf_path: str) -> List[Dict[str, Any]]:
    """
    PDF se sab content extract karke chunks banayein
    
    Args:
        pdf_path: PDF file ka path
    
    Returns:
        List of processed chunks with embeddings
    """
    all_chunks = []
    
    #using recursive character to split the text doc into chunks.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,      # Overlap between chunks (20% of chunk_size)
        length_function=len,    # Function to measure chunk length
        separators=[           # Try splitting by these in order:
            "\n\n",            # 1. Double newlines (paragraphs) - best
            "\n",              # 2. Single newlines (lines)
            ". ",              # 3. Sentences
            " ",               # 4. Words
            ""                 # 5. Characters (last resort)
        ]
    )
        
    # Step 1: Text extract karein
    logger.info("\n📄 Extracting text...")
    text_chunks, page_text_map = extract_text_from_pdf(pdf_path)
    
    # Step 2: Images extract karein
    logger.info("\n🖼️  Extracting images...")
    image_chunks = extract_images_from_pdf(pdf_path)
    
    # Step 3: Tables extract karein
    logger.info("\n📊 Extracting tables...")
    table_chunks = extract_tables_from_pdf(pdf_path)
    
    # Step 4: Text chunks process karein
    logger.info("\n⚙️  Processing text chunks...")
    
    for chunk in text_chunks:
        # 1️⃣ Page-level text ko aur chhote semantic chunks mein split karte hain
        sub_texts = text_splitter.split_text(chunk['content'])

        for sub_text in sub_texts:

            # 2️⃣ Har small text chunk ko LLM se summarize / normalize karte hain
            processed_text = process_text_with_closedmodel(sub_text)

            # 3️⃣ Summarized text ka embedding banate hain
            embedding = generate_embeddings(processed_text)

            # 4️⃣ Final chunk ko store karte hain
            # NOTE:
            # Recursive splitting and parent-child metadata are applied ONLY to text content
            # because text pages can contain multiple semantic topics.
            # Images and tables are treated as atomic semantic units:
            # - An image caption already represents the full meaning of the image
            # - A table’s insight depends on the complete table, not individual rows
            # Therefore, images and tables are directly embedded without further splitting.
            all_chunks.append({
                "type": "text",
                "processed_content": processed_text,
                "embedding": embedding,
                "page_number": chunk["page_number"],
                "source": chunk["source"]
            })

        
    
    
    # Step 5: Image chunks process karein (open-source model se)
    print("\n⚙️  Processing image chunks...")
    for chunk in image_chunks:
        description = generate_image_description_opensource(chunk['image_path'])
        
        enriched = enrich_image_caption_dynamic(
                description,
                chunk["page_number"],
                page_text_map
            )
        all_chunks.append({
            "type": "image",
            "processed_content": enriched,
            "embedding": generate_embeddings(enriched),
            "page_number": chunk["page_number"],
            "source": chunk["source"]
        })
        
        
        
    
    # Step 6: Table chunks process karein
    print("\n⚙️  Processing table chunks...")
    for chunk in table_chunks:
        processed_table = process_table_with_closedmodel(chunk['content'])
        all_chunks.append({
            "type": "table",
            "processed_content": processed_table,
            "embedding": generate_embeddings(processed_table),
            "page_number": chunk["page_number"],
            "source": chunk["source"]
        })

    
    print(f"\n✅ Created {len(all_chunks)} total chunks")
    return all_chunks

In [28]:
# ============================================
# BLOCK 9: PINECONE INDEX SETUP FUNCTION
# ============================================

from pinecone import Pinecone, ServerlessSpec
import time

def get_or_create_pinecone_index(
    api_key: str,
    index_name: str,
    dimension: int = 1536,
    metric: str = "dotproduct",
    cloud: str = "aws",
    region: str = "us-east-1",
    wait_time: int = 10
):
    """
    Create Pinecone index if it does not exist and return index object.

    This function ensures:
    - Index is created ONLY once
    - Index exists BEFORE vector ingestion
    - Correct dimension & metric are enforced
    """

    pc = Pinecone(api_key=api_key)

    if not pc.has_index(index_name):
        print(f"🆕 Creating Pinecone index: {index_name}")

        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric=metric,
            spec=ServerlessSpec(
                cloud=cloud,
                region=region
            )
        )

        print("⏳ Waiting for index to be ready...")
        time.sleep(wait_time)

    else:
        print(f"✅ Using existing Pinecone index: {index_name}")

    index = pc.Index(index_name)

    # Optional sanity check
    stats = index.describe_index_stats()
    print("\n📊 Pinecone Index Stats:")
    print(f"  - Dimension      : {stats.dimension}")
    print(f"  - Vector Count   : {stats.total_vector_count}")
    print(f"  - Index Fullness : {stats.index_fullness}")

    return index


In [29]:
# # BLOCK 10: STORE IN PINECONE(This Function Only used to store Dense Vector)
# # ============================================
# def store_in_pinecone(index, chunks: List[Dict[str, Any]], batch_size: int = 100):
#     """
#     Chunks ko Pinecone database mein store karein
    
#     Args:
#         index: Pinecone index object
#         chunks: List of chunks with embeddings
#         batch_size: Batch size for upsert
#     """
#     print("\n💾 Storing chunks in Pinecone...")
    
#     # Chunks ko batches mein divide karein
#     for i in range(0, len(chunks), batch_size):
#         batch = chunks[i:i + batch_size]
        
#         # Pinecone ke liye vectors prepare karein
#         vectors = []
#         for idx, chunk in enumerate(batch):
#             vector_id = f"chunk_{i + idx}"
            
#             # Metadata prepare karein (embedding ko chhod kar)
#             metadata = {
#                 'type': chunk['type'],
#                 'page_number': chunk['page_number'],
#                 'source': chunk['source'],
#                 'content': chunk.get('processed_content', '')[:1000]  # Pinecone metadata limit
#             }
            
#             # Vector tuple create karein: (id, embedding, metadata)
#             vectors.append((vector_id, chunk['embedding'], metadata))
        
#         # Batch upsert karein
#         index.upsert(vectors=vectors)
#         logger.info(f"✅ Uploaded batch {i // batch_size + 1}")
    
#     logger.info(f"\n✅ Successfully stored {len(chunks)} chunks in Pinecone")

In [30]:
# ============================================
# BLOCK For Creating sparse Matrix That will help in Keyward search operation: BM25 SPARSE ENCODER SETUP
# ============================================

from pinecone_text.sparse import BM25Encoder

def train_bm25_encoder(chunks, save_path="multimodalbm25_encoder.json"):
    """
    Train BM25 encoder on document corpus
    """

    # Corpus = processed textual content
    texts = [chunk["processed_content"] for chunk in chunks]
    

    logger.info(f"📚 Training BM25 on {len(texts)} documents...")
    
    bm25 = BM25Encoder()
    bm25.fit(texts)

    bm25.dump(save_path)
    logger.info("✅ BM25 encoder trained & saved")

    return bm25


In [31]:
# ============================================
# BLOCK For PineHybrid retriever setup: CREATE HYBRID RETRIEVER
# ============================================

from langchain_community.retrievers import PineconeHybridSearchRetriever

def create_hybrid_retriever(index, embeddings, bm25_encoder):
    """
    Create Pinecone hybrid retriever (dense + sparse)
    """
    hybrid_retriever = PineconeHybridSearchRetriever(
        embeddings=emb_model,           # Dense embeddings (semantic searching)
        sparse_encoder=bm25_encoder,    # Sparse encoder (keyword searching)
        index=index,                    # Pinecone index name
        top_k=30,                       # Retrieve top 20 candidates (before reranking)
                                        # Higher value = more comprehensive but slower
        alpha=0.5                       # Balance between dense and sparse
                                        # 0.0 = pure keyword (BM25 only)
                                        # 0.5 = balanced (50% semantic, 50% keyword)
                                        # 1.0 = pure semantic (embeddings only)
                                        # Recommendation: 0.5 for general, 0.3 for technical
    )

    logger.info(f"✅ Hybrid retriever created")
    logger.info(f"  - Top-k: 20 (candidates before reranking)")
    logger.info(f"  - Alpha: 0.5 (balanced semantic + keyword)")
    return hybrid_retriever


In [32]:
# ============================================
# BLOCK: STORE VECTORS (DENSE + SPARSE)
# ============================================

from uuid import uuid4
import time

def store_hybrid_vectors(retriever, chunks, batch_size=50):
    """
    Store dense + sparse vectors with COMPLETE metadata
    """
    texts = []
    metadatas = []
    ids = []

    for chunk in chunks:
        texts.append(chunk["processed_content"])

        metadatas.append({
            "type": chunk["type"],              # text / image / table
            "page_number": chunk["page_number"],
            "source": chunk["source"]
        })

        ids.append(str(uuid4()))

    logger.info("💾 Storing vectors into Pinecone...")

    for i in range(0, len(texts), batch_size):
        retriever.add_texts(
            texts=texts[i:i + batch_size],
            metadatas=metadatas[i:i + batch_size],
            ids=ids[i:i + batch_size]
        )

        logger.info(f"✓ Stored {min(i + batch_size, len(texts))}/{len(texts)}")

    time.sleep(2)
    stats = retriever.index.describe_index_stats()
    logger.info(f"\n📊 Total vectors stored: {stats.total_vector_count}")


In [33]:
# ============================================
# BLOCK: COHERE RERANK SETUP
# ============================================

from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank

def create_reranked_retriever(hybrid_retriever):
    """
    Wrap Pinecone Hybrid Retriever with Cohere Reranking
    """

    compressor = CohereRerank(
        model="rerank-english-v3.0",   # Latest Cohere cross-encoder
        top_n=10,                       # Final documents after reranking
        cohere_api_key=COHERE_API_KEY
    )

    reranked_retriever = ContextualCompressionRetriever(
        base_retriever=hybrid_retriever,
        base_compressor=compressor
    )

    logger.info("✅ Cohere Reranking layer attached")
    logger.info("  - Hybrid retrieval: top 20")
    logger.info("  - Reranked output : top 5")

    return reranked_retriever


In [ ]:
# ============================================
# BLOCK: MAIN PIPELINE(That will extract image,text,table --->convert to embedding vector and store into pinecone database)
# ============================================

def process_pdf_pipeline(pdf_path: str):
    logger.info("=" * 60)
    logger.info("🚀 STARTING PDF PROCESSING PIPELINE")
    logger.info("=" * 60)

    # 1️⃣ Create / Get Pinecone index (BEFORE ingestion)
    index = get_or_create_pinecone_index(
        api_key=PINECONE_API_KEY,
        index_name=INDEX_NAME,
        dimension=1536,
        metric="dotproduct",
        cloud="aws",
        region="us-east-1"
    )

    # 2️⃣ Create document chunks (text split + summarize + embed)
    chunks = create_document_chunks(pdf_path)
    
    
     # 3️⃣ Train BM25
    bm25_encoder = train_bm25_encoder(chunks)
    
    
    # 4️⃣ Create hybrid retriever
    hybrid_retriever = create_hybrid_retriever(
        index=index,
        embeddings=emb_model,   # OpenAIEmbeddings
        bm25_encoder=bm25_encoder
    )

    # 5️⃣ Store dense + sparse vectors
    store_hybrid_vectors(
        retriever=hybrid_retriever,
        chunks=chunks
    )
    
    

    logger.info("\n✅ HYBRID + RERANK PIPELINE COMPLETED")



In [ ]:
from pathlib import Path

def process_pdf_directory(root_dir: str):
    root_path = Path(root_dir)

    for pdf_path in root_path.rglob("*.pdf"):
        logger.info(f"\n📄 Processing: {pdf_path}")
        reranked_retriever = process_pdf_pipeline(str(pdf_path))
        



In [36]:
reranked_retriever = process_pdf_directory(
    r"D:\Langchain_LangGraph_03-12-2025\PDFFolder"
)

reranked_retriever

[2025-12-29 15:58:38,697]-2648173226.py-INFO -
📄 Processing: D:\Langchain_LangGraph_03-12-2025\PDFFolder\attention.pdf
[2025-12-29 15:58:38,698]-1014323218.py-INFO -============================================================
[2025-12-29 15:58:38,700]-1014323218.py-INFO -🚀 STARTING PDF PROCESSING PIPELINE
[2025-12-29 15:58:38,701]-1014323218.py-INFO -============================================================
🆕 Creating Pinecone index: research-papers-rerank-index
⏳ Waiting for index to be ready...

📊 Pinecone Index Stats:
  - Dimension      : 1536
  - Vector Count   : 0
  - Index Fullness : 0.0
[2025-12-29 15:58:56,209]-2155891549.py-INFO -
📄 Extracting text...
[2025-12-29 15:58:56,354]-387780913.py-INFO -📄 Total pages found: 15
✅ Extracted text from 15 pages
[2025-12-29 15:58:59,354]-2155891549.py-INFO -
🖼️  Extracting images...
✅ Extracted 3 images
[2025-12-29 15:58:59,638]-2155891549.py-INFO -
📊 Extracting tables...
[2025-12-29 15:58:59,641]-161720935.py-INFO -📊 Extracting tables 

  0%|          | 0/132 [00:00<?, ?it/s]

[2025-12-29 16:03:38,393]-603021576.py-INFO -✅ BM25 encoder trained & saved
[2025-12-29 16:03:38,400]-1230818308.py-INFO -✅ Hybrid retriever created
[2025-12-29 16:03:38,402]-1230818308.py-INFO -  - Top-k: 20 (candidates before reranking)
[2025-12-29 16:03:38,403]-1230818308.py-INFO -  - Alpha: 0.5 (balanced semantic + keyword)
[2025-12-29 16:03:38,404]-1247981085.py-INFO -💾 Storing vectors into Pinecone...


  0%|          | 0/2 [00:00<?, ?it/s]

[2025-12-29 16:03:39,089]-_client.py-INFO -HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2025-12-29 16:03:45,632]-_client.py-INFO -HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2025-12-29 16:03:51,530]-1247981085.py-INFO -✓ Stored 50/132


  0%|          | 0/2 [00:00<?, ?it/s]

[2025-12-29 16:03:52,197]-_client.py-INFO -HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2025-12-29 16:04:04,786]-_client.py-INFO -HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2025-12-29 16:04:13,103]-1247981085.py-INFO -✓ Stored 100/132


  0%|          | 0/1 [00:00<?, ?it/s]

[2025-12-29 16:04:13,694]-_client.py-INFO -HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2025-12-29 16:04:28,675]-1247981085.py-INFO -✓ Stored 132/132
[2025-12-29 16:04:30,991]-1247981085.py-INFO -
📊 Total vectors stored: 132
[2025-12-29 16:04:30,993]-1014323218.py-INFO -
✅ HYBRID + RERANK PIPELINE COMPLETED


# below code we are checking at time of user query how the document is retrieve and give answer to user

In [62]:
def load_reranked_retriever_for_query():
    """
    Query time pe retriever create hota hai
    """
    # 1️⃣ Pinecone index load
    index = get_or_create_pinecone_index(
        api_key=PINECONE_API_KEY,
        index_name=INDEX_NAME,
        dimension=1536,
        metric="dotproduct",
        cloud="aws",
        region="us-east-1"
    )

    # 2️⃣ BM25 encoder load (jo ingestion time pe save hua tha)
    bm25_encoder = BM25Encoder().load("multimodalbm25_encoder.json")

    # 3️⃣ Hybrid retriever
    hybrid_retriever = create_hybrid_retriever(
        index=index,
        embeddings=emb_model,
        bm25_encoder=bm25_encoder
    )

    # 4️⃣ Cohere rerank
    reranked_retriever = create_reranked_retriever(hybrid_retriever)

    return reranked_retriever


In [63]:
# ===============================
# 1️⃣ LOAD RETRIEVER (QUERY TIME)
# ===============================

from pinecone_text.sparse import BM25Encoder

def load_reranked_retriever_for_query():
    """
    Query time pe retriever create hota hai
    (INGESTION ke baad)
    """

    # Load existing Pinecone index
    index = get_or_create_pinecone_index(
        api_key=PINECONE_API_KEY,
        index_name=INDEX_NAME,
        dimension=1536,
        metric="dotproduct",
        cloud="aws",
        region="us-east-1"
    )

    # Load saved BM25 encoder
    bm25_encoder = BM25Encoder().load(
        "multimodalbm25_encoder.json"
    )

    # Hybrid retriever (dense + sparse)
    hybrid_retriever = create_hybrid_retriever(
        index=index,
        embeddings=emb_model,
        bm25_encoder=bm25_encoder
    )

    # Cohere reranker
    reranked_retriever = create_reranked_retriever(
        hybrid_retriever
    )

    return reranked_retriever


# 🔥 Retriever ko ek hi baar load karo
RERANKED_RETRIEVER = load_reranked_retriever_for_query()




✅ Using existing Pinecone index: research-papers-rerank-index

📊 Pinecone Index Stats:
  - Dimension      : 1536
  - Vector Count   : 132
  - Index Fullness : 0.0
[2025-12-29 16:57:49,294]-1230818308.py-INFO -✅ Hybrid retriever created
[2025-12-29 16:57:49,296]-1230818308.py-INFO -  - Top-k: 20 (candidates before reranking)
[2025-12-29 16:57:49,298]-1230818308.py-INFO -  - Alpha: 0.5 (balanced semantic + keyword)
[2025-12-29 16:57:49,600]-3681714021.py-INFO -✅ Cohere Reranking layer attached
[2025-12-29 16:57:49,601]-3681714021.py-INFO -  - Hybrid retrieval: top 20
[2025-12-29 16:57:49,602]-3681714021.py-INFO -  - Reranked output : top 5


In [65]:
# ===============================
# 2️⃣ QUERY INTENT DETECTION
# ===============================

def detect_query_intent(query: str):
    q = query.lower()

    if any(word in q for word in ["diagram", "architecture", "figure", "image"]):
        return "image"

    if any(word in q for word in ["table", "rows", "columns", "compare"]):
        return "table"

    return "text"


In [66]:
# ===============================
# 3️⃣ SMART RETRIEVAL (IMAGE / TABLE)
# ===============================

def retrieve_with_modality_bias(retriever, query):
    """
    User ke sawal ke hisaab se
    image / table ko priority deta hai
    """

    intent = detect_query_intent(query)

    # Pinecone se docs lao
    docs = retriever.get_relevant_documents(query)

    # Agar image ya table specifically poocha gaya ho
    if intent in ["image", "table"]:
        filtered_docs = [
            d for d in docs
            if d.metadata.get("type") == intent
        ]

        # Agar matching docs mile
        if filtered_docs:
            return filtered_docs

    # Fallback: sab kuch
    return docs


In [69]:
# ===============================
# 4️⃣ FORMAT CONTEXT FOR LLM
# ===============================

def format_docs(docs):
    """
    LLM ko clearly batata hai:
    - yeh IMAGE hai
    - yeh TABLE hai
    - yeh TEXT hai
    """

    formatted_chunks = []

    for doc in docs:
        print(doc)
        print("*"*50)
        doc_type = doc.metadata.get("type", "text").upper()
        page = doc.metadata.get("page_number", "NA")

        formatted_chunks.append(
            f"[TYPE: {doc_type} | PAGE: {page}]\n{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)




In [70]:
# ===============================
# 5️⃣ BUILD CONTEXT (HELPER)
# ===============================

def build_context(user_query: str) -> str:
    docs = retrieve_with_modality_bias(
        RERANKED_RETRIEVER,
        user_query
    )
    return format_docs(docs)




In [71]:
#augumenting question and context together.
#converting dynamic query to structure instruction prompt
prompt = PromptTemplate(
    template="""
You are an expert multimodal RAG assistant that can understand TEXT, IMAGES, and TABLES.

Context (contains text, image descriptions, and tables):
{context}

Question:
{question}

Instructions:
1. Use ONLY the context above to answer the question
2. For IMAGE-based questions:
   - Describe the image content from the description
   - Reference the image location if asked
3. For TABLE-based questions:
   - Present the table data clearly
   - Explain relationships in the table
4. For TEXT-based questions:
   - Answer concisely using the text content
5. If the answer requires multiple modalities (e.g., "explain the diagram and related table"), 
   combine information from all relevant sources
6. If information is missing, say: "I don't know based on the provided context"
7. Always cite the PAGE NUMBER and TYPE of source used

Answer:
""",
    input_variables=["context", "question"]
)

In [72]:
def query_rag(user_query: str):

    reranked_retriever = load_reranked_retriever_for_query()

    parallel_chain = RunnableParallel(
        context=RunnableLambda(
            lambda q: format_docs(
                retrieve_with_modality_bias(reranked_retriever, q)
            )
        ),
        question=RunnablePassthrough()
    )

    rag_chain = parallel_chain | prompt | model1 | parser

    return rag_chain.invoke(user_query)


In [73]:
query = """
Explain the Transformer architecture diagram and encoder-decoder flow
"""

response = query_rag(query)
print(response)

✅ Using existing Pinecone index: research-papers-rerank-index

📊 Pinecone Index Stats:
  - Dimension      : 1536
  - Vector Count   : 132
  - Index Fullness : 0.0
[2025-12-29 17:00:41,352]-1230818308.py-INFO -✅ Hybrid retriever created
[2025-12-29 17:00:41,355]-1230818308.py-INFO -  - Top-k: 20 (candidates before reranking)
[2025-12-29 17:00:41,356]-1230818308.py-INFO -  - Alpha: 0.5 (balanced semantic + keyword)
[2025-12-29 17:00:41,620]-3681714021.py-INFO -✅ Cohere Reranking layer attached
[2025-12-29 17:00:41,621]-3681714021.py-INFO -  - Hybrid retrieval: top 20
[2025-12-29 17:00:41,622]-3681714021.py-INFO -  - Reranked output : top 5
[2025-12-29 17:00:42,575]-_client.py-INFO -HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2025-12-29 17:00:43,912]-_client.py-INFO -HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"
page_content='The diagram illustrates the architecture of a Transformer model, showcasing stacked self-attention and fully c

In [74]:
query2 = """
Show and explain the Transformer architecture diagram from the 
"Attention Is All You Need" paper and explain each block briefly.
"""

response2 = query_rag(query2)
print(response2)


✅ Using existing Pinecone index: research-papers-rerank-index

📊 Pinecone Index Stats:
  - Dimension      : 1536
  - Vector Count   : 132
  - Index Fullness : 0.0
[2025-12-29 17:01:03,163]-1230818308.py-INFO -✅ Hybrid retriever created
[2025-12-29 17:01:03,166]-1230818308.py-INFO -  - Top-k: 20 (candidates before reranking)
[2025-12-29 17:01:03,168]-1230818308.py-INFO -  - Alpha: 0.5 (balanced semantic + keyword)
[2025-12-29 17:01:03,398]-3681714021.py-INFO -✅ Cohere Reranking layer attached
[2025-12-29 17:01:03,398]-3681714021.py-INFO -  - Hybrid retrieval: top 20
[2025-12-29 17:01:03,399]-3681714021.py-INFO -  - Reranked output : top 5
[2025-12-29 17:01:03,973]-_client.py-INFO -HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2025-12-29 17:01:05,105]-_client.py-INFO -HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"
page_content='The diagram illustrates the architecture of a Transformer model, showcasing stacked self-attention and fully c

In [75]:
query3 = "Describe the diagram shown in the paper"
print(query_rag(query3))

✅ Using existing Pinecone index: research-papers-rerank-index

📊 Pinecone Index Stats:
  - Dimension      : 1536
  - Vector Count   : 132
  - Index Fullness : 0.0
[2025-12-29 17:01:10,166]-1230818308.py-INFO -✅ Hybrid retriever created
[2025-12-29 17:01:10,168]-1230818308.py-INFO -  - Top-k: 20 (candidates before reranking)
[2025-12-29 17:01:10,170]-1230818308.py-INFO -  - Alpha: 0.5 (balanced semantic + keyword)
[2025-12-29 17:01:10,446]-3681714021.py-INFO -✅ Cohere Reranking layer attached
[2025-12-29 17:01:10,447]-3681714021.py-INFO -  - Hybrid retrieval: top 20
[2025-12-29 17:01:10,448]-3681714021.py-INFO -  - Reranked output : top 5
[2025-12-29 17:01:10,948]-_client.py-INFO -HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
[2025-12-29 17:01:11,922]-_client.py-INFO -HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"
page_content='The diagram illustrates the architecture of a Transformer model, showcasing stacked self-attention and fully c